In [91]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [92]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)
pd.options.display.float_format = '{:.2f}'.format

1. Collecting raw data

The six main factors, effecting the risk of an asthma excerbation, investigated in this prototype are: Particulate Matter <= 2.5 micrograms (PM2.5), nitrogen dioxide (NO2), ozone (O3), pollen, relative Humidity and temperature. Raw data is collected from Open Meteo.

In [93]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": 34.80,
	"longitude": 38.99,
	"hourly": ["pm2_5", "nitrogen_dioxide", "ozone",],
    "timezone": "auto",
	"past_days": 1,
}
air_quality_responses = openmeteo.weather_api(url, params = params)


Defining a python list with of the hourly and current raw data values.

In [94]:
# Process first location. Add a for-loop for multiple locations or weather models
airQualityResponse = air_quality_responses[0]
# Process hourly data. The order of variables needs to be the same as requested.
hourly_AQ = airQualityResponse.Hourly()
hourly_pm2_5 = hourly_AQ.Variables(0).ValuesAsNumpy()
hourly_nitrogen_dioxide = hourly_AQ.Variables(1).ValuesAsNumpy()
hourly_ozone = hourly_AQ.Variables(2).ValuesAsNumpy()

hourly_air_quality_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly_AQ.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly_AQ.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly_AQ.Interval()),
		inclusive = "left"
	).tz_convert(airQualityResponse.Timezone().decode())
}

hourly_air_quality_data["pm2_5"] = hourly_pm2_5
hourly_air_quality_data["nitrogen_dioxide"] = hourly_nitrogen_dioxide
hourly_air_quality_data["ozone"] = hourly_ozone

hourly_air_quality_dataframe = pd.DataFrame(data = hourly_air_quality_data)
print("\n This is the Hourly Air Quality Index (AQI) DataFrame:")
hourly_air_quality_dataframe.head(5)


 This is the Hourly Air Quality Index (AQI) DataFrame:


,date,pm2_5,nitrogen_dioxide,ozone
0,2026-09-04 00:00:00+03:00,9.20,0.60,86.00
1,2026-09-04 01:00:00+03:00,11.30,0.60,99.00
2,2026-09-04 02:00:00+03:00,9.50,0.60,90.00
3,2026-09-04 03:00:00+03:00,10.10,0.60,90.00
4,2026-09-04 04:00:00+03:00,9.80,0.70,82.00


In [95]:
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": 34.80,
	"longitude": 38.99,
	"hourly": ["birch_pollen", "grass_pollen", "ragweed_pollen"],
	"timezone": "auto",
	"past_days": 4,
}
pollen_responses = openmeteo.weather_api(url, params = params)

In [96]:
# Process first location. Add a for-loop for multiple locations or weather models
pollenResponse = pollen_responses[0]

# Process hourly data. The order of variables needs to be the same as requested.
hourlyPollen = pollenResponse.Hourly()
hourly_birch_pollen = hourlyPollen.Variables(0).ValuesAsNumpy()
hourly_grass_pollen = hourlyPollen.Variables(1).ValuesAsNumpy()
hourly_ragweed_pollen = hourlyPollen.Variables(2).ValuesAsNumpy()

hourly_pollen_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourlyPollen.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourlyPollen.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourlyPollen.Interval()),
		inclusive = "left"
	).tz_convert(pollenResponse.Timezone().decode())
}

hourly_pollen_data["birch_pollen"] = hourly_birch_pollen
hourly_pollen_data["grass_pollen"] = hourly_grass_pollen
hourly_pollen_data["ragweed_pollen"] = hourly_ragweed_pollen

hourly_pollen_df = pd.DataFrame(data = hourly_pollen_data)
hourly_pollen_df.head(20)

,date,birch_pollen,grass_pollen,ragweed_pollen
0,2026-09-01 00:00:00+03:00,0.00,0.50,0.30
1,2026-09-01 01:00:00+03:00,0.00,0.40,0.30
2,2026-09-01 02:00:00+03:00,0.00,0.20,0.20
3,2026-09-01 03:00:00+03:00,0.00,0.20,0.30
4,2026-09-01 04:00:00+03:00,0.00,0.10,0.30
5,2026-09-01 05:00:00+03:00,0.00,0.10,0.30
6,2026-09-01 06:00:00+03:00,0.00,0.10,0.40
7,2026-09-01 07:00:00+03:00,0.00,0.10,0.40
8,2026-09-01 08:00:00+03:00,0.00,0.20,0.40
9,2026-09-01 09:00:00+03:00,0.00,0.20,0.80


In [97]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 34.80,
	"longitude": 38.99,
	"daily": ["temperature_2m_max", "temperature_2m_min"],
	"hourly": "relative_humidity_2m",
	"timezone": "auto",
	"past_days": 2,
}
weather_responses = openmeteo.weather_api(url, params = params)

Spezifizierung von weather_responses als Liste für Max und Min Temperaturen sowie relative Luftfeuchtigkeit.

In [98]:
weatherResponse = weather_responses[0]

# Process hourly data. The order of variables needs to be the same as requested.
hourlyRH = weatherResponse.Hourly()
hourly_relative_humidity_2m = hourlyRH.Variables(0).ValuesAsNumpy()

hourly_RH_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourlyRH.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourlyRH.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourlyRH.Interval()),
		inclusive = "left"
	).tz_convert(weatherResponse.Timezone().decode())
}

hourly_RH_data["relative_humidity_2m"] = hourly_relative_humidity_2m

hourly_RH_dataframe = pd.DataFrame(data = hourly_RH_data)
print("\n This is the Hourly Relative Humidity DataFrame:")
hourly_RH_dataframe.head(5)


 This is the Hourly Relative Humidity DataFrame:


,date,relative_humidity_2m
0,2026-09-03 00:00:00+03:00,22.00
1,2026-09-03 01:00:00+03:00,23.00
2,2026-09-03 02:00:00+03:00,25.00
3,2026-09-03 03:00:00+03:00,26.00
4,2026-09-03 04:00:00+03:00,26.00


In [99]:

# Process daily data. The order of variables needs to be the same as requested.
dailyTemp = weatherResponse.Daily()
daily_temperature_2m_max = dailyTemp.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = dailyTemp.Variables(1).ValuesAsNumpy()

daily_temperature_data = {
	"date": pd.date_range(
		start = pd.to_datetime(dailyTemp.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(dailyTemp.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = dailyTemp.Interval()),
		inclusive = "left"
	).tz_convert(weatherResponse.Timezone().decode())
}

daily_temperature_data["temperature_2m_max"] = daily_temperature_2m_max
daily_temperature_data["temperature_2m_min"] = daily_temperature_2m_min

daily_temp_df = pd.DataFrame(data = daily_temperature_data)
print("\n This is the Daily Maximum and Minimum Temperature DataFrame:")
daily_temp_df.head(5)


 This is the Daily Maximum and Minimum Temperature DataFrame:


,date,temperature_2m_max,temperature_2m_min
0,2026-09-03 00:00:00+03:00,38.18,26.98
1,2026-09-04 00:00:00+03:00,38.63,25.98
2,2026-09-05 00:00:00+03:00,38.08,26.83
3,2026-09-06 00:00:00+03:00,34.73,25.83
4,2026-09-07 00:00:00+03:00,35.68,26.23


2. Manipulating Data for Asthma Exacerbation Risk calculations

According to studies one cannot rely solely on the current data values to predict the risk of a possible exacerbation. The diural temperature range (DTR) for example is more of an indicating factor than the raw current temperature value. The mean of the last 24 hours of air quality factors (nitrogen dioxide and PM2.5) and last 8 hours of ozone should be taken into account. As for the relative humidity the mean as well as the difference value of the last 24 hours have to be calculated. The difference value is by definition the current mean minus the previous mean of the same time period (24 hours). Generally, an asthma exacerbation depends mostly on the daily concentration of pollen with a lag of 0-4 days. 

In [100]:
hourly_air_quality_dataframe['Moving PM2.5 mean'] = hourly_air_quality_dataframe['pm2_5'].rolling(window = 24).mean()
hourly_air_quality_dataframe['Moving NO2 mean'] = hourly_air_quality_dataframe['nitrogen_dioxide'].rolling(window = 24).mean()
hourly_air_quality_dataframe['Moving O3 mean'] = hourly_air_quality_dataframe['ozone'].rolling(window = 8).mean()
hourly_air_quality_dataframe.head()

,date,pm2_5,nitrogen_dioxide,ozone,Moving PM2.5 mean,Moving NO2 mean,Moving O3 mean
0,2026-09-04 00:00:00+03:00,9.20,0.60,86.00,NaN,NaN,NaN
1,2026-09-04 01:00:00+03:00,11.30,0.60,99.00,NaN,NaN,NaN
2,2026-09-04 02:00:00+03:00,9.50,0.60,90.00,NaN,NaN,NaN
3,2026-09-04 03:00:00+03:00,10.10,0.60,90.00,NaN,NaN,NaN
4,2026-09-04 04:00:00+03:00,9.80,0.70,82.00,NaN,NaN,NaN


In [101]:
# Calculate the current PM2.5 mean over the last 24 hours
current_time = pd.Timestamp.now(hourly_air_quality_dataframe['date'].dt.tz)
print(current_time)
current_row = hourly_air_quality_dataframe[
    (hourly_air_quality_dataframe['date'].dt.date == current_time.date())&
    (hourly_air_quality_dataframe['date'].dt.hour == current_time.hour)
    ]
current_PM2_5_Mean = current_row['Moving PM2.5 mean'].iloc[0]
current_NO2_Mean = current_row['Moving NO2 mean'].iloc[0]
current_O3_Mean = current_row['Moving O3 mean'].iloc[0]
print(f"\n This is the current PM2.5 mean of the last 24 hours: {current_PM2_5_Mean:.2f}")
print(f"\n This is the current NO2 mean of the last 24 hours: {current_NO2_Mean:.2f}")
print(f"\n This is the current O3 mean of the last 8 hours: {current_O3_Mean:.2f}")

2026-09-05 13:28:36.726532+03:00

 This is the current PM2.5 mean of the last 24 hours: 14.05

 This is the current NO2 mean of the last 24 hours: 0.38

 This is the current O3 mean of the last 8 hours: 96.12


In [102]:
# Calculating:
# 1. the 72-hour mean for birch pollen,
hourly_pollen_df['Birch Pollen 72H mean'] = hourly_pollen_df['birch_pollen'].rolling(window = 72).mean()
# 2. 24-hour mean (daily) and shift for grass pollen,
hourly_pollen_df['Grass Pollen 24H mean'] = hourly_pollen_df['grass_pollen'].rolling(window = 24).mean()
hourly_pollen_df['Grass Pollen 72H Shift'] = hourly_pollen_df['Grass Pollen 24H mean'].shift(72)
# 3. and 72-hour mean for ragweed pollen:
hourly_pollen_df['Ragweed Pollen 72H mean'] = hourly_pollen_df['ragweed_pollen'].rolling(window = 72).mean()
hourly_pollen_df.iloc[96:110]

,date,birch_pollen,grass_pollen,ragweed_pollen,Birch Pollen 72H mean,Grass Pollen 24H mean,Grass Pollen 72H Shift,Ragweed Pollen 72H mean
96,2026-09-05 00:00:00+03:00,0.00,0.60,0.10,0.00,0.23,0.20,0.22
97,2026-09-05 01:00:00+03:00,0.00,0.60,0.10,0.00,0.24,0.20,0.22
98,2026-09-05 02:00:00+03:00,0.00,0.50,0.10,0.00,0.25,0.20,0.21
99,2026-09-05 03:00:00+03:00,0.00,0.40,0.10,0.00,0.25,0.21,0.21
100,2026-09-05 04:00:00+03:00,0.00,0.30,0.10,0.00,0.25,0.21,0.21
101,2026-09-05 05:00:00+03:00,0.00,0.30,0.10,0.00,0.26,0.22,0.21
102,2026-09-05 06:00:00+03:00,0.00,0.20,0.10,0.00,0.26,0.22,0.21
103,2026-09-05 07:00:00+03:00,0.00,0.20,0.10,0.00,0.26,0.23,0.21
104,2026-09-05 08:00:00+03:00,0.00,0.20,0.20,0.00,0.26,0.23,0.21
105,2026-09-05 09:00:00+03:00,0.00,0.20,0.10,0.00,0.26,0.23,0.21


In [114]:
current_time_Pollen = pd.Timestamp.now(hourly_pollen_df['date'].dt.tz)
print(f"\n Current local time: {current_time_Pollen}")
current_row_pollen = hourly_pollen_df[
    (hourly_pollen_df['date'].dt.date == current_time_Pollen.date())&
    (hourly_pollen_df['date'].dt.hour == current_time_Pollen.hour)
]
# Assigning Variable names for the required variables to be used in the risk assessment model. The variable names are self-explanatory.
Birch_Pollen_72H_mean = current_row_pollen['Birch Pollen 72H mean'].iloc[0]
print(f"\n This is the current Birch Pollen mean of the last 72 hours: {Birch_Pollen_72H_mean:.2f}")

Grass_Pollen_72H_lag = current_row_pollen['Grass Pollen 72H Shift'].iloc[0]
print(f"\n This is the mean Grass 24 Hour Pollen concentration with a lag of 3 days: {Grass_Pollen_72H_lag:.2f}")

Ragweed_Pollen_72H_mean = current_row_pollen['Ragweed Pollen 72H mean'].iloc[0]
print(f"\n This is the current Ragweed Pollen mean of the last 72 hours: {Ragweed_Pollen_72H_mean:.2f}")
current_row_pollen


 Current local time: 2026-09-05 13:37:31.826010+03:00

 This is the current Birch Pollen mean of the last 72 hours: 0.00

 This is the mean Grass 24 Hour Pollen concentration with a lag of 3 days: 0.23

 This is the current Ragweed Pollen mean of the last 72 hours: 0.20


,date,birch_pollen,grass_pollen,ragweed_pollen,Birch Pollen 72H mean,Grass Pollen 24H mean,Grass Pollen 72H Shift,Ragweed Pollen 72H mean
109,2026-09-05 13:00:00+03:00,0.00,0.10,0.10,0.00,0.26,0.23,0.20


In [104]:
hourly_RH_dataframe['Moving Relative Humidity mean'] = hourly_RH_dataframe['relative_humidity_2m'].rolling(window = 24).mean()
hourly_RH_dataframe['Shifted Moving Relative Humidity mean'] = hourly_RH_dataframe['Moving Relative Humidity mean'].shift(24)
print(f"\n DF showing mean and the preceding mean of Relative Humidity of 24 hours timeframe:")
hourly_RH_dataframe.loc[45:72]


 DF showing mean and the preceding mean of Relative Humidity of 24 hours timeframe:


,date,relative_humidity_2m,Moving Relative Humidity mean,Shifted Moving Relative Humidity mean
45,2026-09-04 21:00:00+03:00,18.00,23.25,NaN
46,2026-09-04 22:00:00+03:00,20.00,23.17,NaN
47,2026-09-04 23:00:00+03:00,28.00,23.42,21.62
48,2026-09-05 00:00:00+03:00,49.00,24.50,21.67
49,2026-09-05 01:00:00+03:00,52.00,25.62,21.75
50,2026-09-05 02:00:00+03:00,53.00,26.71,21.83
51,2026-09-05 03:00:00+03:00,53.00,27.75,21.92
52,2026-09-05 04:00:00+03:00,55.00,28.88,22.00
53,2026-09-05 05:00:00+03:00,54.00,29.92,22.04
54,2026-09-05 06:00:00+03:00,55.00,30.79,22.29


In [124]:
# Calculating current local time and the corresponding row in the hourly_RH_dataframe to get the current
# 24-hour mean of Relative Humidity and the previous 24-hour mean of Relative Humidity. Then, calculating
# the difference between the two means.
current_time_RH = pd.Timestamp.now(hourly_RH_dataframe['date'].dt.tz)
print(f"\n Current local time: {current_time_RH}")
current_row_RH = hourly_RH_dataframe[
    (hourly_RH_dataframe['date'].dt.date == current_time_RH.date())&
    (hourly_RH_dataframe['date'].dt.hour == current_time_RH.hour)
]
current_24_RH_Mean = current_row_RH['Moving Relative Humidity mean'].iloc[0]
prev_24_RH_Mean = current_row_RH['Shifted Moving Relative Humidity mean'].iloc[0]
print(f"\n This is the current 24-hour mean of Relative Humidity: {current_24_RH_Mean:.2f}")
print(f"\n This is the previous 24-hour mean of Relative Humidity: {prev_24_RH_Mean:.2f}")
mean_RH_difference = current_24_RH_Mean - prev_24_RH_Mean
print(f"\n The difference between the current 24-hour mean and the previous 24-hour mean of Relative Humidity is: {mean_RH_difference:.2f}")


 Current local time: 2026-09-05 14:58:05.287077+03:00

 This is the current 24-hour mean of Relative Humidity: 35.12

 This is the previous 24-hour mean of Relative Humidity: 23.83

 The difference between the current 24-hour mean and the previous 24-hour mean of Relative Humidity is: 11.29


In [106]:
daily_temp_df['DTR'] = daily_temp_df['temperature_2m_max'] - daily_temp_df['temperature_2m_min']
print(f"\n DF showing the daily maximum and minimum temperature and the difference between them:")
daily_temp_df.head()


 DF showing the daily maximum and minimum temperature and the difference between them:


,date,temperature_2m_max,temperature_2m_min,DTR
0,2026-09-03 00:00:00+03:00,38.18,26.98,11.20
1,2026-09-04 00:00:00+03:00,38.63,25.98,12.65
2,2026-09-05 00:00:00+03:00,38.08,26.83,11.25
3,2026-09-06 00:00:00+03:00,34.73,25.83,8.90
4,2026-09-07 00:00:00+03:00,35.68,26.23,9.45


In [107]:
current_date = pd.Timestamp.now(daily_temp_df['date'].dt.tz).date()
print(f"\n Current date: {current_date}")
current_row_temp = daily_temp_df[daily_temp_df['date'].dt.date == current_date]
current_temp_max = current_row_temp['temperature_2m_max'].iloc[0]
current_temp_min = current_row_temp['temperature_2m_min'].iloc[0]
current_temp_diff = current_temp_max - current_temp_min
print(f"\n Current maximum temperature: {current_temp_max:.2f}")
print(f"\n Current minimum temperature: {current_temp_min:.2f}")
print(f"\n Diurnal Temperature Range: {current_temp_diff:.2f}")


 Current date: 2026-09-05

 Current maximum temperature: 38.08

 Current minimum temperature: 26.83

 Diurnal Temperature Range: 11.25


3. Final Data Collection 

Collect all relevant Environmental Factors and their corresponding Values in one Dataset.

In [125]:
environmental_factors_df = pd.DataFrame({
    'Current PM2.5 Mean': [current_PM2_5_Mean],
    'Current NO2 Mean': [current_NO2_Mean],
    'Current O3 Mean': [current_O3_Mean],
    'Birch Pollen 72H Mean': [Birch_Pollen_72H_mean],
    'Grass Pollen 72H Lag': [Grass_Pollen_72H_lag],
    'Ragweed Pollen 72H Mean': [Ragweed_Pollen_72H_mean],
    'Mean RH Difference': [mean_RH_difference],
    'Diurnal Temp Range': [current_temp_diff]},
    index = [current_row_RH['date'].iloc[0]]
)
# environmental_factors_df = pd.DataFrame(data, index = ['current_date'])
environmental_factors_df

,Current PM2.5 Mean,Current NO2 Mean,Current O3 Mean,Birch Pollen 72H Mean,Grass Pollen 72H Lag,Ragweed Pollen 72H Mean,Mean RH Difference,Diurnal Temp Range
2026-09-05 14:00:00+03:00,14.05,0.38,96.12,0.00,0.23,0.20,11.29,11.25
